# Polaris Results

Inspect local Polaris runs under `data/polaris_results/`, log in to the Hub, and
upload results. The runner saves metrics and unlabeled test predictions; test
targets are never read here. Uploads reuse the `BenchmarkResults` already
saved in each run's `metadata.json`, so scores match `metrics.csv` exactly.

Produce runs first, from a representation environment:

```bash
micromamba activate minimol
python scripts/run_polaris_benchmark.py --config configs/polaris.yaml \
    --dataset pgp_broccatelli --representation minimol
```

Inspect and upload from any environment with `polaris` installed (`minimol`,
`chemeleon`, `chemprop`) and the repo editable-installed:

```bash
micromamba activate minimol
pip install -e .
```

Uploads are one-way: leave `UPLOAD = False` until you are ready to push.

In [ ]:
from pathlib import Path
import json

import pandas as pd

ROOT = Path.cwd()
ARTIFACTS = ROOT / "data" / "polaris_results"
if not ARTIFACTS.exists():
    raise FileNotFoundError(f"No local runs found: {ARTIFACTS}")

UPLOAD = True  
OWNER = "brunoliva" 

## Local runs

In [2]:
def load_runs(root=ARTIFACTS):
    rows = []
    for metadata_path in sorted(root.glob("*/*/metadata.json")):
        metadata = json.loads(metadata_path.read_text())
        metrics = pd.read_csv(metadata_path.parent / "metrics.csv")
        for row in metrics.to_dict("records"):
            rows.append({**metadata, **row, "artifact_dir": str(metadata_path.parent)})
    return pd.DataFrame(rows)


runs = load_runs()
runs[
    [
        "benchmark",
        "representation",
        "metric",
        "Metric",
        "Score",
        "n_train",
        "n_test",
        "artifact_dir",
    ]
]

,benchmark,representation,metric,Metric,Score,n_train,n_test,artifact_dir
0,tdcommons/ames,chemeleon,roc_auc,roc_auc,0.869322,5821,1457,/home/bruno/brood/data/polaris_results/ames/ta...
1,tdcommons/ames,minimol,roc_auc,roc_auc,0.870837,5821,1457,/home/bruno/brood/data/polaris_results/ames/ta...
2,tdcommons/bbb-martins,chemeleon,roc_auc,roc_auc,0.911077,1624,406,/home/bruno/brood/data/polaris_results/bbb_mar...
3,tdcommons/bbb-martins,minimol,roc_auc,roc_auc,0.928549,1624,406,/home/bruno/brood/data/polaris_results/bbb_mar...
4,tdcommons/bioavailability-ma,chemeleon,roc_auc,roc_auc,0.694047,512,128,/home/bruno/brood/data/polaris_results/bioavai...
5,tdcommons/bioavailability-ma,minimol,roc_auc,roc_auc,0.722315,512,128,/home/bruno/brood/data/polaris_results/bioavai...
6,tdcommons/caco2-wang,chemeleon,mean_absolute_error,mean_absolute_error,0.285720,728,182,/home/bruno/brood/data/polaris_results/caco2_w...
7,tdcommons/caco2-wang,minimol,mean_absolute_error,mean_absolute_error,0.308534,728,182,/home/bruno/brood/data/polaris_results/caco2_w...
8,tdcommons/clearance-hepatocyte-az,chemeleon,spearmanr,spearmanr,0.428611,970,243,/home/bruno/brood/data/polaris_results/clearan...
9,tdcommons/clearance-hepatocyte-az,minimol,spearmanr,spearmanr,0.454467,970,243,/home/bruno/brood/data/polaris_results/clearan...


## Login

The token is cached on disk, so later runs of this notebook reuse it. Re-run
this cell in a fresh kernel if it expires.

In [ ]:
from polaris.hub.client import PolarisHubClient

client = PolarisHubClient()
client.login()
me = client.external_client.user_info
username = me.get("username") or me.get("email") or "unknown"
print(f"Logged in as {username}")

/home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/authlib/integrations/httpx_client/assertion_client.py:5: AuthlibDeprecationWarning: The httpx module is deprecated; please use httpx2 instead.
  from ._compat import httpx2


[2026-10-08 09:34:18 PM] INFO     You are successfully logged in to the Polaris Hub.                  ]8;id=1058186;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/hub/client.py\client.py]8;;\:]8;id=1058187;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/hub/client.py#245\245]8;;\

Logged in as brunoholiva (set OWNER above if it is still a placeholder)


## Check before uploading

Each run's `metadata.json` holds the full `BenchmarkResults`: the scores the
official protocol produced, plus the name, description and tags that go to the
Hub. The cell below prints exactly what would be sent, per run, without sending
anything.

The Hub exposes no list-my-results API, so after an upload the only
confirmation is the result URL each upload prints.

In [4]:
from polaris.evaluate import BenchmarkResultsV1

GITHUB_URL = None  


def submit(dataset, representation):
    """Rebuild one saved run's results and upload it when UPLOAD is True."""
    run_dir = ARTIFACTS / dataset / f"tabpfn_{representation}"
    meta = json.loads((run_dir / "metadata.json").read_text())
    results = BenchmarkResultsV1.model_validate(meta["result"])
    results.user_attributes = {
        "Framework": "TabPFN v3.5",
        "Representation": representation,
    }
    if GITHUB_URL:
        results.github_url = GITHUB_URL
    scores = ", ".join(
        f"{row['Metric']}={row['Score']:.4f}"
        for _, row in results.results.iterrows()
    )
    if not UPLOAD:
        print(f"[dry run] {results.benchmark_artifact_id} {results.name}: {scores}")
        return results
    if not OWNER or OWNER.startswith("<"):
        raise ValueError("Set OWNER to your Polaris username before uploading.")
    client.upload_results(results, owner=OWNER)
    print(f"uploaded {results.benchmark_artifact_id} {results.name}: {scores}")
    return results

## Upload

One cell per dataset, one plain call per representation — nothing loops. Run a
cell with `UPLOAD = False` first to read the dry-run lines, then flip the flag
and run only the datasets you want to push. Re-running a cell after an upload
sends a duplicate, so treat each dataset as a one-shot.

In [ ]:
# tdcommons/ames
submit("ames", "chemeleon")
submit("ames", "minimol")


In [ ]:
# tdcommons/bbb-martins
submit("bbb_martins", "chemeleon")
submit("bbb_martins", "minimol")


In [ ]:
# tdcommons/bioavailability-ma
submit("bioavailability_ma", "chemeleon")
submit("bioavailability_ma", "minimol")


In [ ]:
# tdcommons/caco2-wang
submit("caco2_wang", "chemeleon")
submit("caco2_wang", "minimol")


In [ ]:
# tdcommons/clearance-hepatocyte-az
submit("clearance_hepatocyte_az", "chemeleon")
submit("clearance_hepatocyte_az", "minimol")


In [ ]:
# tdcommons/clearance-microsome-az
submit("clearance_microsome_az", "chemeleon")
submit("clearance_microsome_az", "minimol")


In [ ]:
# tdcommons/cyp2c9-substrate-carbonmangels
submit("cyp2c9_substrate_carbonmangels", "chemeleon")
submit("cyp2c9_substrate_carbonmangels", "minimol")


In [ ]:
# tdcommons/cyp2c9-veith
submit("cyp2c9_veith", "chemeleon")
submit("cyp2c9_veith", "minimol")


In [ ]:
# tdcommons/cyp2d6-substrate-carbonmangels
submit("cyp2d6_substrate_carbonmangels", "chemeleon")
submit("cyp2d6_substrate_carbonmangels", "minimol")


In [ ]:
# tdcommons/cyp2d6-veith
submit("cyp2d6_veith", "chemeleon")
submit("cyp2d6_veith", "minimol")


In [ ]:
# tdcommons/cyp3a4-substrate-carbonmangels
submit("cyp3a4_substrate_carbonmangels", "chemeleon")
submit("cyp3a4_substrate_carbonmangels", "minimol")


In [ ]:
# tdcommons/cyp3a4-veith
submit("cyp3a4_veith", "chemeleon")
submit("cyp3a4_veith", "minimol")


In [ ]:
# tdcommons/half-life-obach
submit("half_life_obach", "chemeleon")
submit("half_life_obach", "minimol")


In [ ]:
# tdcommons/herg
submit("herg", "chemeleon")
submit("herg", "minimol")


In [5]:
# tdcommons/hia-hou
submit("hia_hou", "chemeleon")
submit("hia_hou", "minimol")


/home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

[09:34:46 PM] Your result has been successfully uploaded to the Hub. View it here:                    ]8;id=1058193;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/hub/client.py\client.py]8;;\:]8;id=1058194;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/hub/client.py#527\527]8;;\
              https://polarishub.io/benchmarks/tdcommons/hia-hou/ZPqS6SqE6yAEAAydTTNWP                             

               Success: Uploading results                                                             ]8;id=1058201;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/utils/context.py\context.py]8;;\:]8;id=1058202;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/utils/context.py#53\53]8;;\

/home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

uploaded tdcommons/hia-hou tabpfn-chemeleon: roc_auc=0.9856


[09:34:47 PM] Your result has been successfully uploaded to the Hub. View it here:                    ]8;id=1058207;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/hub/client.py\client.py]8;;\:]8;id=1058208;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/hub/client.py#527\527]8;;\
              https://polarishub.io/benchmarks/tdcommons/hia-hou/1sFYRR3hcXsu9bGnFBVAq                             

               Success: Uploading results                                                             ]8;id=1058213;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/utils/context.py\context.py]8;;\:]8;id=1058214;file:///home/bruno/micromamba/envs/brood/lib/python3.11/site-packages/polaris/utils/context.py#53\53]8;;\

uploaded tdcommons/hia-hou tabpfn-minimol: roc_auc=0.9951


{
  "benchmark_artifact_id": "tdcommons/hia-hou",
  "benchmark_name": null,
  "benchmark_owner": null,
  "results": [
    {
      "test_set": "test",
      "target_label": "Y",
      "scores": {
        "roc_auc": 0.9950617283950618
      }
    }
  ],
  "name": "tabpfn-minimol",
  "description": "TabPFN v3.5 + minimol.",
  "tags": [
    "tabpfn",
    "minimol"
  ],
  "user_attributes": {
    "Framework": "TabPFN v3.5",
    "Representation": "minimol"
  },
  "owner": "brunoliva",
  "polaris_version": "0.13.0",
  "slug": "tabpfn-minimol",
  "github_url": null,
  "paper_url": null,
  "contributors": [],
  "artifact_id": "brunoliva/tabpfn-minimol",
  "urn": "urn:polaris:result:brunoliva:tabpfn-minimol",
  "version": 1
}

In [ ]:
# tdcommons/ld50-zhu
submit("ld50_zhu", "chemeleon")
submit("ld50_zhu", "minimol")


In [ ]:
# tdcommons/lipophilicity-astrazeneca
submit("lipophilicity_astrazeneca", "chemeleon")
submit("lipophilicity_astrazeneca", "minimol")


In [ ]:
# tdcommons/pgp-broccatelli
submit("pgp_broccatelli", "chemeleon")
submit("pgp_broccatelli", "minimol")


In [ ]:
# tdcommons/ppbr-az
submit("ppbr_az", "chemeleon")
submit("ppbr_az", "minimol")


In [ ]:
# tdcommons/vdss-lombardo
submit("vdss_lombardo", "chemeleon")
submit("vdss_lombardo", "minimol")
